In [161]:
from __future__ import annotations

from __future__ import annotations

import argparse
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
import logging
import os
import pathlib
import re
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import mplhep as mh
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import torch
import torch.nn.functional as F
import yaml

# Configuração de semente aleatória e estilo visual ATLAS para todos os plots
np.random.seed(42)
mh.style.use("ATLAS")

from src.core.Trainers.NeuralRingerTrainer import NeuralRingerTrainer
from src.Parser.NeuralRingerTrainerConfiguration import NeuralRingerTrainerConfiguration
from src.utils import create_folder, norm1

# Configuração de Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# Plotter Utils

In [162]:
def compute_pearson_with_uncertainty(
    y_A: np.ndarray, y_B: np.ndarray, confidence_level: float = 0.95
) -> Tuple[float, float, float, Tuple[float, float]]:
    """Calcula o coeficiente de Pearson r, p-valor, incerteza e intervalo de confiança (IC 95%) via transformada Z de Fisher.

    Args:
        y_A: Primeiro perfil de energia (1D array).
        y_B: Segundo perfil de energia (1D array).
        confidence_level: Nível de confiança estatística (padrão: 0.95 para 95%).

    Returns:
        Tuple contendo: (coef_pearson, p_valor, incerteza, (ci_low, ci_high))
    """
    from scipy.stats import norm

    res = pearsonr(y_A, y_B)
    if hasattr(res, "statistic"):
        r = float(res.statistic)
        p_val = float(res.pvalue)
    else:
        r, p_val = float(res[0]), float(res[1])

    N = len(y_A)
    if N <= 3:
        return r, p_val, 0.0, (r, r)

    # Transformada Z de Fisher para cálculo da incerteza e intervalo de confiança
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))
    se_z = 1.0 / np.sqrt(N - 3)

    z_crit = norm.ppf(1.0 - (1.0 - confidence_level) / 2.0)
    z_low, z_high = z - z_crit * se_z, z + z_crit * se_z

    ci_low, ci_high = np.tanh(z_low), np.tanh(z_high)
    uncertainty = (ci_high - ci_low) / 2.0

    return r, p_val, float(uncertainty), (float(ci_low), float(ci_high))



In [163]:
def get_config(config_path: str) -> NeuralRingerTrainerConfiguration:
    """Lê o arquivo YAML e retorna a configuração validada.

    Args:
        config_path: Caminho para o arquivo de configuração YAML.

    Returns:
        NeuralRingerTrainerConfiguration: Instância validada da configuração.
    """
    logger.info("Carregando configuração de: %s", config_path)
    try:
        with open(config_path, "r", encoding="utf-8") as file:
            yaml_data = yaml.safe_load(file)
            return NeuralRingerTrainerConfiguration.model_validate(yaml_data)
    except FileNotFoundError:
        logger.error("Arquivo de configuração não encontrado: %s", config_path)
        raise
    except Exception as e:
        logger.error("Erro ao ler o arquivo de configuração: %s", e)
        raise

In [164]:
def group_and_concat_results(directory_path: pathlib.Path) -> Dict[str, pd.DataFrame]:
    """Agrupa arquivos .pkl buscando pelo padrão 'ietX.ietaY' no nome.

    Gera um dicionário mapeando o nome da região para um DataFrame concatenado.

    Args:
        directory_path: Diretório contendo os arquivos de resultados .pkl.

    Returns:
        Dict[str, pd.DataFrame]: Mapeamento de region_key -> DataFrame agrupado.
    """
    if not directory_path.exists():
        raise FileNotFoundError(f"Diretório de resultados não existe: {directory_path}")

    logger.info("Lendo diretório de resultados: %s", directory_path)
    files = [p for p in directory_path.iterdir() if p.is_file() and p.suffix == ".pkl"]

    groups = defaultdict(list)
    for file_path in files:
        match = re.search(r"(iet\d+\.ieta\d+)", file_path.name)
        group_key = match.group(1) if match else "undefined_region"
        groups[group_key].append(file_path)

    grouped_dfs = {}
    for group_key, file_list in groups.items():
        logger.info(" -> Encontrados %d arquivos para a região %s. Concatenando...", len(file_list), group_key)
        df_list = [pd.read_pickle(f) for f in file_list]
        grouped_dfs[group_key] = pd.concat(df_list, ignore_index=True)

    return grouped_dfs

In [165]:
def extract_convolutional_features(model: Any, x: torch.Tensor) -> torch.Tensor:
    """Extrai a saída das camadas convolucionais de um modelo PyTorch.

    Args:
        model: Modelo PyTorch com camadas conv1 e conv2.
        x: Tensor de entrada contendo as feições dos anéis.

    Returns:
        torch.Tensor: Tensor de feições após conv1, conv2 e ReLU.
    """
    x = x.view(-1, 1, model.input_dim)
    x = F.relu(model.conv1(x))
    x = F.relu(model.conv2(x))
    return x


In [166]:
def compute_saliency_map(x: torch.Tensor, model: torch.nn.Module) -> np.ndarray:
    """Calcula o mapa de saliência normalizado via gradiente para uma amostra.

    Args:
        x: Tensor 1D contendo as feições de entrada de uma amostra.
        model: Modelo PyTorch avaliado.

    Returns:
        np.ndarray: Array contendo a saliência normalizada [0, 1].
    """
    x_tensor = x.unsqueeze(0).clone().detach().requires_grad_(True)
    pred = model(x_tensor)
    grad = torch.autograd.grad(outputs=pred.sum(), inputs=x_tensor)[0]
    saliency = np.abs(grad.cpu().numpy()[0])
    smin, smax = saliency.min(), saliency.max()
    return (saliency - smin) / (smax - smin + 1e-8)

In [167]:
def compute_mean_std_saliency(
    X_subset: torch.Tensor, model: torch.nn.Module
) -> Tuple[np.ndarray, np.ndarray]:
    """Calcula a média e o desvio padrão da saliência para um conjunto de dados.

    Args:
        X_subset: Tensor contendo múltiplas amostras de entrada.
        model: Modelo PyTorch.

    Returns:
        Tuple[np.ndarray, np.ndarray]: (média da saliência, desvio padrão).
    """
    saliency_list = [compute_saliency_map(x, model) for x in X_subset]
    saliency_array = np.stack(saliency_list)
    return np.mean(saliency_array, axis=0), np.std(saliency_array, axis=0)

# dataclass

In [168]:
@dataclass
class RegionPlotContext:
    """Objeto de contexto que transporta dados e metadados de uma região cinemática."""

    iet: int
    ieta: int
    output_dir: pathlib.Path
    data: np.ndarray
    target: np.ndarray
    test_indices: np.ndarray
    trainer: Any
    model: torch.nn.Module

    @property
    def x_holdout_full(self) -> np.ndarray:
        return self.data[self.test_indices]

    @property
    def y_holdout(self) -> np.ndarray:
        return self.target[self.test_indices]

    @property
    def x_holdout_rings(self) -> np.ndarray:
        return self.x_holdout_full[:, self.trainer.full_dataset.ring_column_indices]

    @property
    def ring_indexes(self) -> np.ndarray:
        return self.trainer.full_dataset.indexes

In [169]:
class BasePlotter(ABC):
    """Classe base abstrata para geradores de gráficos.

    Novos métodos de plotagem devem herdar desta classe e implementar o método `plot`.
    """

    def __init__(self, name: str) -> None:
        self.name = name

    @abstractmethod
    def plot(self, context: RegionPlotContext, **kwargs) -> None:
        """Método principal de execução da plotagem.

        Args:
            context: Instância de RegionPlotContext com os dados da região.
            **kwargs: Parâmetros adicionais customizados.
        """
        pass

    @staticmethod
    def save_figure(
        fig: plt.Figure,
        folder_path: str | pathlib.Path,
        subfolder: str,
        filename: str
    ) -> str:
        """Salva uma figura Matplotlib em PDF e limpa a memória com segurança.

        Args:
            fig: Instância da figura Matplotlib.
            folder_path: Diretório base.
            subfolder: Nome da pasta temática (ex.: 'RingsMeanProfiles').
            filename: Nome do arquivo PDF gerado.

        Returns:
            str: Caminho absoluto onde o gráfico foi salvo.
        """
        try:
            path = create_folder(subfolder, str(folder_path))
            save_path = os.path.join(path, filename)
            fig.savefig(
                save_path,
                format="pdf",
                dpi=300,
                transparent=True,
                bbox_inches="tight",
            )
            logger.info("Gráfico salvo em: %s", save_path)
            return save_path
        finally:
            plt.close(fig)

In [170]:
class ProfileMeanEnergyPlotter(BasePlotter):
    """Gera o perfil médio de energia nos anéis para Sinal (Fótons) vs Background (Jatos)."""

    def __init__(self) -> None:
        super().__init__(name="ProfileMeanEnergyPlotter")

    def plot(
        self,
        context: RegionPlotContext,
        x_rings: Optional[np.ndarray] = None,
        plot_name: str = "RingsMeanProfiles_NeuralRinger",
        plot_title: str = "Average Energy Profile in the Rings - NeuralRinger",
        **kwargs
    ) -> None:
        rings_data = x_rings if x_rings is not None else context.x_holdout_rings
        y_holdout = context.y_holdout
        ring_index = context.ring_indexes

        logger.info("[%s] Gerando perfil médio para matriz de forma %s...", self.name, rings_data.shape)

        n_signal = np.sum(y_holdout == 1)
        n_jets = np.sum(y_holdout == 0)
        logger.info("Amostras - Sinal: %d | Jatos: %d", n_signal, n_jets)

        normalized_rings = norm1(rings_data)
        signal_data = normalized_rings[y_holdout == 1]
        bg_data = normalized_rings[y_holdout == 0]

        n_rings = signal_data.shape[1]
        x_axis = np.arange(n_rings)

        mean_signal, std_signal = np.mean(signal_data, axis=0), np.std(signal_data, axis=0)
        mean_bkg, std_bkg = np.mean(bg_data, axis=0), np.std(bg_data, axis=0)
        y_max = max(np.max(mean_bkg), np.max(mean_signal))

        def get_ring_pos(val: int) -> int:
            idx = np.where(np.atleast_1d(ring_index) == val)[0]
            return int(idx[0]) if len(idx) > 0 else 0

        subdet_names = ["PreSampler", "EM1", "EM2", "EM3", "TileCal"]
        subdet_x = [
            0,
            get_ring_pos(8),
            get_ring_pos(72),
            get_ring_pos(80),
            get_ring_pos(88),
        ]
        subdet_colors = ["#1b9e77", "#d95f09", "#7570b3", "#e7298a", "#66a61e"]

        fig, ax = plt.subplots(figsize=(10, 5), clear=True, num=1)

        ax.errorbar(
            x_axis, mean_signal, yerr=std_signal,
            marker="o", mfc="navy", mec="navy", ms=3,
            mew=0.5, elinewidth=0.8, capsize=2,
            ecolor="navy", color="navy", label="Photons"
        )
        ax.errorbar(
            x_axis, mean_bkg, yerr=std_bkg,
            marker="s", mfc="darkorange", mec="darkorange", ms=3,
            mew=0.5, elinewidth=0.8, capsize=2,
            ecolor="darkorange", color="darkorange", label="Hadronic Jets"
        )

        for x_pos, name, color in zip(subdet_x, subdet_names, subdet_colors):
            ax.axvline(x=x_pos, color=color, linestyle="--", linewidth=1)
            ax.text(
                x_pos + 1.2, y_max * 1.2, name, rotation=90,
                va="bottom", ha="center", fontsize=9, color=color
            )

        ax.set_xlabel("Rings", fontsize=13)
        ax.set_ylabel("Normalized Energy", fontsize=13)

        ticks_loc = np.linspace(0, n_rings, 10, dtype=int)
        ax.set_xticks(ticks_loc)
        ax.set_xticklabels([str(i + 1) for i in ticks_loc], fontsize=11)
        ax.tick_params(axis="y", labelsize=11)

        ax.set_ylim(-0.05, y_max * 1.5)
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend(fontsize=10, loc="upper right")
        ax.set_title(plot_title, fontsize=14)
        fig.tight_layout()

        filename = f"et{context.iet}.eta{context.ieta}.{plot_name}.pdf"
        self.save_figure(fig, context.output_dir, "RingsMeanProfiles", filename)

In [171]:

class PearsonCorrelationPlotter(BasePlotter):
    """Calcula a correlação de Pearson entre o dado de entrada e a saída da camada convolucional separadamente para Sinal (Fótons) e Ruído/Background (Jatos)."""

    def __init__(self) -> None:
        super().__init__(name="PearsonCorrelationPlotter")

    def plot(
        self,
        context: RegionPlotContext,
        y_A_signal: np.ndarray,
        y_B_signal: np.ndarray,
        y_A_bkg: np.ndarray,
        y_B_bkg: np.ndarray,
        plot_name: str = "pearson_correlation",
        confidence_level: float = 0.95,
        **kwargs
    ) -> Dict[str, Tuple[float, float, float, Tuple[float, float]]]:
        """Calcula a correlação de Pearson e incertezas para Sinal (Fótons) e Ruído (Jatos) separadamente.

        Returns:
            Dict contendo os resultados estatísticos por classe ('signal' e 'background').
        """
        r_sig, p_sig, unc_sig, (ci_low_sig, ci_high_sig) = compute_pearson_with_uncertainty(
            y_A_signal, y_B_signal, confidence_level
        )
        r_bkg, p_bkg, unc_bkg, (ci_low_bkg, ci_high_bkg) = compute_pearson_with_uncertainty(
            y_A_bkg, y_B_bkg, confidence_level
        )

        logger.info(
            "[%s] iet=%d, ieta=%d | Sinal (Fótons): r=%.4f±%.4f | Ruído (Jatos): r=%.4f±%.4f",
            self.name, context.iet, context.ieta, r_sig, unc_sig, r_bkg, unc_bkg
        )

        n_rings = len(y_A_signal)
        x_axis = np.arange(n_rings)

        fig, ax = plt.subplots(figsize=(10, 6), clear=True, num=1)

        # Curvas do Sinal (Fótons)
        ax.plot(x_axis, y_A_signal, label="Entrada - Photons", color="navy", linewidth=2, linestyle="-")
        ax.plot(x_axis, y_B_signal, label=f"Conv - Photons (r = {r_sig:.3f}±{unc_sig:.3f})", color="navy", linewidth=1.5, linestyle="--")

        # Curvas do Ruído / Background (Jatos Hadrônicos)
        ax.plot(x_axis, y_A_bkg, label="Entrada - Hadronic Jets", color="darkorange", linewidth=2, linestyle="-")
        ax.plot(x_axis, y_B_bkg, label=f"Conv - Hadronic Jets (r = {r_bkg:.3f}±{unc_bkg:.3f})", color="darkorange", linewidth=1.5, linestyle="--")

        ax.set_title(
            f"Pearson: Entrada vs Conv Layer (Sinal vs Ruído) [iet={context.iet}, ieta={context.ieta}]\n"
            f"Photons: r = {r_sig:.3f}±{unc_sig:.3f} | Jets: r = {r_bkg:.3f}±{unc_bkg:.3f}"
        )
        ax.set_xlabel("Rings", fontsize=13)
        ax.set_ylabel("Energia Normalizada", fontsize=13)

        ticks_loc = np.linspace(0, n_rings, 10, dtype=int)
        ax.set_xticks(ticks_loc)
        ax.set_xticklabels([str(i + 1) for i in ticks_loc], fontsize=11)

        ax.legend(fontsize=9, loc="upper right")
        ax.grid(True, linestyle=":", alpha=0.7)
        fig.tight_layout()

        filename = f"iet{context.iet}.ieta{context.ieta}_{plot_name}.pdf"
        self.save_figure(fig, context.output_dir, "pearson", filename)

        return {
            "signal": (r_sig, p_sig, unc_sig, (ci_low_sig, ci_high_sig)),
            "background": (r_bkg, p_bkg, unc_bkg, (ci_low_bkg, ci_high_bkg)),
        }


In [172]:
class ConvLayerPlotter(BasePlotter):
    """Avalia o modelo PyTorch e plota as feições extraídas da camada convolucional."""

    def __init__(self) -> None:
        super().__init__(name="ConvLayerPlotter")
        self.profile_plotter = ProfileMeanEnergyPlotter()
        self.pearson_plotter = PearsonCorrelationPlotter()

    def plot(
        self,
        context: RegionPlotContext,
        plot_title: str = "Feature Extraction - Conv layer output",
        **kwargs
    ) -> None:
        logger.info("[%s] Avaliando modelo convolucional na região iet=%d, ieta=%d...", self.name, context.iet, context.ieta)
        context.model.eval()

        y_holdout = context.y_holdout
        X_holdout_rings = context.x_holdout_rings
        input_rings_norm = norm1(X_holdout_rings)

        input_signal = input_rings_norm[y_holdout == 1]
        input_bkg = input_rings_norm[y_holdout == 0]

        mean_input_signal = np.mean(input_signal, axis=0)
        mean_input_bkg = np.mean(input_bkg, axis=0)

        tensor_in = torch.from_numpy(X_holdout_rings).float().to(context.trainer.device)

        with torch.no_grad():
            saida_conv_tensor = extract_convolutional_features(context.model, tensor_in)

        saida_conv_np = saida_conv_tensor.squeeze().cpu().numpy()

        for ch in range(4):
            if saida_conv_np.ndim == 3:
                feature_slice = saida_conv_np[:, ch, :]
            elif saida_conv_np.ndim == 2:
                feature_slice = saida_conv_np
            else:
                logger.warning("Shape inesperado da camada conv: %s", saida_conv_np.shape)
                feature_slice = saida_conv_np

            # 1. Perfil médio das feições extraídas do canal (separado por classe)
            self.profile_plotter.plot(
                context=context,
                x_rings=feature_slice,
                plot_name=f"ConvFeatureExtraction_plot_ringer.ch{ch}",
                plot_title=f"{plot_title}.ch{ch}"
            )

            # 2. Correlação de Pearson por classe (Photons vs Hadronic Jets)
            conv_norm = norm1(feature_slice)
            conv_signal = conv_norm[y_holdout == 1]
            conv_bkg = conv_norm[y_holdout == 0]

            mean_conv_signal = np.mean(conv_signal, axis=0)
            mean_conv_bkg = np.mean(conv_bkg, axis=0)

            self.pearson_plotter.plot(
                context=context,
                y_A_signal=mean_input_signal,
                y_B_signal=mean_conv_signal,
                y_A_bkg=mean_input_bkg,
                y_B_bkg=mean_conv_bkg,
                plot_name=f"pearson_input_vs_conv_ch{ch}"
            )



In [173]:
class SaliencyMapPlotter(BasePlotter):
    """Calcula e plota a comparação de mapas de saliência por gradiente autograd."""

    def __init__(self) -> None:
        super().__init__(name="SaliencyMapPlotter")

    def plot(
        self,
        context: RegionPlotContext,
        n_samples: int = 1000,
        **kwargs
    ) -> None:
        logger.info("[%s] Gerando gráfico de saliência para %d amostras...", self.name, n_samples)
        
        y_test = context.y_holdout
        X_test = torch.from_numpy(context.x_holdout_rings).float().to(context.trainer.device)

        idx_sinal = np.where(y_test == 1)[0]
        idx_backg = np.where(y_test == 0)[0]
        np.random.shuffle(idx_sinal)
        np.random.shuffle(idx_backg)

        idx_sinal = idx_sinal[:n_samples]
        idx_backg = idx_backg[:n_samples]

        X_sinal = X_test[idx_sinal]
        X_backg = X_test[idx_backg]

        mean_sinal, std_sinal = compute_mean_std_saliency(X_sinal, context.model)
        mean_backg, std_backg = compute_mean_std_saliency(X_backg, context.model)

        fig, ax = plt.subplots(figsize=(12, 6), clear=True, num=1)
        x_range = np.arange(len(mean_sinal))

        ax.plot(mean_sinal, label="Signal (mean)", color="blue")
        ax.fill_between(
            x_range,
            mean_sinal - std_sinal,
            mean_sinal + std_sinal,
            color="blue",
            alpha=0.3,
            label="Signal ±1σ",
        )
        ax.plot(mean_backg, label="Background (mean)", color="red")
        ax.fill_between(
            x_range,
            mean_backg - std_backg,
            mean_backg + std_backg,
            color="red",
            alpha=0.3,
            label="Background ±1σ",
        )

        ax.set_title(
            f"Comparison of Normalized Mean Saliency Profile\n"
            f"({n_samples} samples per class)\n({context.iet},{context.ieta})"
        )
        ax.set_xlabel("Input Position - Ring Index")

        n_rings = len(mean_sinal)
        ticks_loc = np.linspace(0, n_rings, 10, dtype=int)
        ax.set_xticks(ticks_loc)
        ax.set_xticklabels([str(i + 1) for i in ticks_loc])

        ax.set_ylabel("Normalized Importance (Saliency)")
        ax.legend()
        ax.grid(True, linestyle="--")
        fig.tight_layout()

        filename = f"iet{context.iet}.ieta{context.ieta}_saliency_comparison_normalized_{n_rings}Rings.pdf"
        self.save_figure(fig, context.output_dir, "saliency", filename)



# Base

In [174]:
class PlotManager:
    """Gerenciador central de plotters. Permite registrar e executar novos gráficos facilmente."""

    def __init__(self) -> None:
        self._plotters: List[BasePlotter] = []

    def register_plotter(self, plotter: BasePlotter) -> None:
        """Adiciona um novo plotter ao pipeline de execução.

        Args:
            plotter: Instância de uma subclasse de BasePlotter.
        """
        logger.info("Registrando novo plotter no pipeline: %s", plotter.name)
        self._plotters.append(plotter)

    def run_all(self, context: RegionPlotContext, **kwargs) -> None:
        """Executa sequencialmente todos os plotters registrados.

        Args:
            context: Instância do contexto da região cinemática.
            **kwargs: Parâmetros compartilhados.
        """
        logger.info("Iniciando execução do pipeline de gráficos (%d plotters registrados)...", len(self._plotters))
        for plotter in self._plotters:
            try:
                plotter.plot(context, **kwargs)
            except Exception as e:
                logger.error("Erro ao executar o plotter '%s': %s", plotter.name, e, exc_info=True)


In [175]:
def main(
    config: str,
    data_path: str,
    selection_criteria: str = "best_repeat",
    first_only: bool = False,
    target_region: Optional[str] = None,
) -> None:
    """Função principal de análise e geração de gráficos por região cinemática."""
    config_instance = get_config(config)
    trainer = NeuralRingerTrainer(config_instance)
    model = trainer.initModel()

    path_dir = pathlib.Path(data_path)
    plot_dir = pathlib.Path("Plots") / path_dir.name
    plot_dir.mkdir(parents=True, exist_ok=True)

    grouped_dfs = group_and_concat_results(path_dir)

    # Configuração do Gerenciador de Plotters (Adicione novos plotters aqui)
    plot_manager = PlotManager()
    plot_manager.register_plotter(ConvLayerPlotter())
    plot_manager.register_plotter(SaliencyMapPlotter())

    processed_count = 0
    for region_name, df in grouped_dfs.items():
        if region_name == "undefined_region":
            continue

        if target_region and region_name != target_region:
            continue

        logger.info("=" * 60)
        logger.info("Analisando região cinemática: %s (Total de modelos: %d)", region_name, len(df))
        logger.info("=" * 60)

        match = re.search(r"iet(\d+)\.ieta(\d+)", region_name)
        iet = int(match.group(1)) if match else 1
        ieta = int(match.group(2)) if match else 1

        file_name_pattern = f"consolidated.et{iet}.eta{ieta}.npz"
        all_paths = np.array([item for item in trainer.full_dataset.file_paths])
        matched_indices = np.where(np.char.find(all_paths, file_name_pattern) != -1)[0]

        if len(matched_indices) > 0:
            target_index = matched_indices[0]
            data, target, _ = trainer.full_dataset[target_index]
            test_data, _ = trainer.generate_folds_with_holdout(data, target)
        else:
            logger.warning("Arquivo com padrão '%s' não encontrado. Pulo de região...", file_name_pattern)
            continue

        df_stats = df.groupby("reapet")["best_sp_value"].agg(["mean", "std"]).reset_index()

        best_idx = df_stats["mean"].idxmax()
        best_repeat = df_stats.loc[best_idx, "reapet"]
        best_mean_sp = df_stats.loc[best_idx, "mean"]
        std_sp = df_stats.loc[best_idx, "std"]

        logger.info("-----> Melhor Inicialização (repeat): %d | Média SP: %.4f ± %.4f", best_repeat, best_mean_sp, std_sp)

        df_best_repeat = df[df["reapet"] == best_repeat]
        best_repeat_model_idx = df_best_repeat["best_sp_value"].idxmax()
        best_repeat_model = df.loc[best_repeat_model_idx]

        best_overall_model_idx = df["best_sp_value"].idxmax()
        best_overall_model = df.loc[best_overall_model_idx]

        if selection_criteria == "best_repeat":
            logger.info(
                "Carregando pesos do melhor repeat (Repeat: %s, Fold: %s, SP: %.4f)",
                best_repeat_model["reapet"], best_repeat_model["fold"], best_repeat_model["best_sp_value"]
            )
            model.load_state_dict(best_repeat_model["best_weights"])
        else:
            logger.info(
                "Carregando pesos do melhor modelo absoluto (Repeat: %s, Fold: %s, SP: %.4f)",
                best_overall_model["reapet"], best_overall_model["fold"], best_overall_model["best_sp_value"]
            )
            model.load_state_dict(best_overall_model["best_weights"])

        model.to(trainer.device)

        # Construção do objeto de contexto para a região
        context = RegionPlotContext(
            iet=iet,
            ieta=ieta,
            output_dir=plot_dir,
            data=data,
            target=target,
            test_indices=test_data,
            trainer=trainer,
            model=model,
        )

        # Execução automatizada de todos os gráficos via PlotManager
        plot_manager.run_all(context)

        processed_count += 1

        if first_only and not target_region:
            logger.info("Primeira região cinemática processada. Finalizando devido à flag first_only=True.")
            break

    if processed_count == 0:
        logger.warning("Nenhuma região foi processada com sucesso.")

In [179]:
main(
    config="config/NeuralRinger/ModelV1_HighBatch_newExtraction_20_regions_100Rings.yaml",
    data_path="results/config_nameModelV1_HighBatch_newExtraction_20_regions_100Rings_id20260629222107",
    selection_criteria="best_repeat",
    first_only=True,
    target_region=None,
)

01:58:25 [INFO] Carregando configuração de: config/NeuralRinger/ModelV1_HighBatch_newExtraction_20_regions_100Rings.yaml
01:58:25 [INFO] Initializing cross validation: StratifiedKFold
01:58:25 [INFO] Creating Dataset...
01:58:25 [INFO] Dataset created.
01:58:25 [INFO] Initializing model
01:58:25 [INFO] Using CUDA; 1 devices
01:58:25 [INFO] Lendo diretório de resultados: results/config_nameModelV1_HighBatch_newExtraction_20_regions_100Rings_id20260629222107
01:58:25 [INFO]  -> Encontrados 45 arquivos para a região iet1.ieta1. Concatenando...
01:58:26 [INFO]  -> Encontrados 45 arquivos para a região iet2.ieta1. Concatenando...
01:58:26 [INFO]  -> Encontrados 45 arquivos para a região iet3.ieta1. Concatenando...
01:58:26 [INFO]  -> Encontrados 45 arquivos para a região iet4.ieta1. Concatenando...
01:58:26 [INFO] Registrando novo plotter no pipeline: ConvLayerPlotter
01:58:26 [INFO] Registrando novo plotter no pipeline: SaliencyMapPlotter
01:58:26 [INFO] ===================================

In [177]:

main(
    config="config/NeuralRinger/ModelV1_HighBatch_newExtraction_20_regions_50Rings.yaml",
    data_path="results/config_nameModelV1_HighBatch_newExtraction_20_regions_50Rings_id20260630043428",
    selection_criteria="best_repeat",
    first_only=False,
    target_region=None,
)


01:55:48 [INFO] Carregando configuração de: config/NeuralRinger/ModelV1_HighBatch_newExtraction_20_regions_50Rings.yaml
01:55:48 [INFO] Initializing cross validation: StratifiedKFold
01:55:48 [INFO] Creating Dataset...
01:55:48 [INFO] Dataset created.
01:55:48 [INFO] Initializing model
01:55:48 [INFO] Using CUDA; 1 devices
01:55:48 [INFO] Lendo diretório de resultados: results/config_nameModelV1_HighBatch_newExtraction_20_regions_50Rings_id20260630043428
01:55:48 [INFO]  -> Encontrados 45 arquivos para a região iet1.ieta1. Concatenando...
01:55:48 [INFO]  -> Encontrados 45 arquivos para a região iet2.ieta1. Concatenando...
01:55:49 [INFO]  -> Encontrados 45 arquivos para a região iet3.ieta1. Concatenando...
01:55:49 [INFO]  -> Encontrados 45 arquivos para a região iet4.ieta1. Concatenando...
01:55:49 [INFO] Registrando novo plotter no pipeline: ConvLayerPlotter
01:55:49 [INFO] Registrando novo plotter no pipeline: SaliencyMapPlotter
01:55:49 [INFO] =====================================

In [178]:

main(
    config="config/NeuralRinger/ModelV1_HighBatch_newExtraction_20_regions_25Rings.yaml",
    data_path="results/config_nameModelV1_HighBatch_newExtraction_20_regions_25Rings_id20260630122538",
    selection_criteria="best_repeat",
    first_only=False,
    target_region=None,
)


01:56:42 [INFO] Carregando configuração de: config/NeuralRinger/ModelV1_HighBatch_newExtraction_20_regions_25Rings.yaml
01:56:42 [INFO] Initializing cross validation: StratifiedKFold
01:56:42 [INFO] Creating Dataset...
01:56:42 [INFO] Dataset created.
01:56:42 [INFO] Initializing model
01:56:42 [INFO] Using CUDA; 1 devices
01:56:42 [INFO] Lendo diretório de resultados: results/config_nameModelV1_HighBatch_newExtraction_20_regions_25Rings_id20260630122538
01:56:42 [INFO]  -> Encontrados 45 arquivos para a região iet1.ieta1. Concatenando...
01:56:43 [INFO]  -> Encontrados 45 arquivos para a região iet2.ieta1. Concatenando...
01:56:43 [INFO]  -> Encontrados 45 arquivos para a região iet3.ieta1. Concatenando...
01:56:43 [INFO]  -> Encontrados 45 arquivos para a região iet4.ieta1. Concatenando...
01:56:43 [INFO] Registrando novo plotter no pipeline: ConvLayerPlotter
01:56:43 [INFO] Registrando novo plotter no pipeline: SaliencyMapPlotter
01:56:43 [INFO] =====================================